# Trend analysis observations Notebook
## Probabilistic method

---

## How to Use This Notebook

**1. Follow the numbered steps in order.**  
Each section builds upon the previous one, from setup, data loading, and climatology computation, to event analysis and visualization.

**2. Look for <font color="orange"> Orange cells  </font> and code cells marked as <font color="lightgreen">##### (User selection) ##### </font>:** 
| <font color="orange"> Orange cells  </font> | <font color="orange"> Need user intervantion </font>|
| ----------- | ----------- |
| <font color="green">**Green cells** </font> | <font color="green">**Run automatically on user input provided in the orange cells and should not be adjusted in most cases** </font>|


**3. Run cells sequentially.**  
Start from the top and execute each cell (`Shift` + `Enter`).  

### <font color="green"> Import require packages </font>

In [ ]:
from datetime import datetime, timedelta
from c3s_event_attribution_tools import *
import xarray as xr
import pandas as pd
import geopandas as gpd
import os
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# import R libraries from WWA
ro.r('''
if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes", repos="https://cloud.r-project.org", quiet=TRUE)
}

# Force install the older, compatible version of gsl
if (!requireNamespace("gsl", quietly = TRUE)) {
    remotes::install_version("gsl", version = "2.1-8", repos = "https://cloud.r-project.org")
}

# Now install rwwa (Maris fork to ensure availablity)
remotes::install_github("maris-development/rwwa@236d9a6b4a201eca1f28001b5535341022f5aeaf") 
''')

rwwa = importr("rwwa")
%load_ext rpy2.ipython

### <font color="orange"> User specifications </font>

#### <font color="orange"> Authentication & File Setup </font>

Action Required: Ensure you have entered your **CDS API Key** in the code cell above.

Storage: Data and results will be saved to: {{your_save_directory}}.

Security: ⚠️ Never share this notebook publicly or commit it to GitHub with your API keys visible.

In [ ]:
# Directory you wish to store output files in. using ../ specifies the parent directory
CURRENT_DIRECTORY = os.getcwd() # do not touch, __file__ specifies the current directory of the file

################# (User selection) ###################
your_save_directory = os.path.abspath(os.path.join(CURRENT_DIRECTORY, "./data"))   # change ../data to your desired directory
your_api_key = ''
######################################################

### <font color="orange"> Choice of parameter </font>

In [ ]:
# Choice of parameter (Tmax, Tmean, Tmin, Precipitation)
################# (User selection) ###################
parameter = "" 
event_end = datetime(, , ) #YYYY,MM,DD 

######################################################
event_start = event_end - timedelta(days=14)
event_year = event_end.year
######################################################

# Get parameter configuration
config = Utils.get_parameter_config(parameter)
value_col = config["value_col"]
y_label = config["y_label"]
unit = config["unit"]
calculation = config["calculation"] 
if parameter in ["Tmax", "Tmean", "Tmin"]:
    variable = "Temperature"
    fit_type = "shift"
    method = "std"
elif parameter == "Precipitation":
    variable = "Precipitation"
    fit_type = "fixeddisp"
    method = "dispersion"

## 3.2 Check (visually) for inhomogeneity of annual event time series

### <font color='green'>File name of the annual time series from the event definition step</font>

In [ ]:
annual_timeseries_load = 'ts_ann_studyregion.nc'

ts_ann_studyregion = xr.open_dataset(os.path.join(your_save_directory, annual_timeseries_load)).to_dataframe().reset_index()
ts_ann_studyregion

- a. Pay special attention to years that might coincide with key events such as the introduction of satellites (1979)
- b. Jumps in the (annual) time series
- c. visually check for a trend or for jumps outside the confidence interval in the scale or dispersion parameter. The uncertainty interval in the plot contains a first order estimate of the uncertainty according to the chi2 quantity.
     - i. For temperature plot running standard deviation with a 15-year moving window and check for large jumps
     - ii. For precipitation plot running dispersion (stdev/mean) with a 15-year moving window and check for large jumps

In [ ]:
ts_ann_studyregion_15y = Process.calculate_rolling_window(gdf=ts_ann_studyregion, value_col=value_col, datetime_col="year",
                                  window=15, method=method, min_periods=1, centering=True, ci=0.95)

ts_ann_studyregion_15y

In [ ]:
Plot.plot_timeserie(data=ts_ann_studyregion_15y, datetime_col="year", value_col=value_col,
               title=f"15 year running {method} of {parameter}", x_label="Date", y_label=method, line_style='solid', ci=True);

## 3.2.d. Decide on which years to use if data is not homogeneous

### <font color='orange'>Specify the preferred time range based on the plot</font>

In [ ]:
year_range = (1950, event_year)

### <font color='green'> Slice the annual time series on the given time range </font>

In [ ]:
ts_ann_studyregion_subset = Utils.subset_gdf(gdf=ts_ann_studyregion, datetime_col="year", date_range=year_range)
ts_ann_studyregion_subset

## 3.3 Conclude about the quality of ERA5
(and potentially other datasets) for the specific event definition and implications for the study results and the years to use, write into the output table in Notes & tables and the scientific report Section 2.1.
- a. For the region, check performance of the dataset, conclude on whether the data can be used with limited problems (“satisfactory”) or caution is necessary (“caution”) and conclude on a sentence in the scientific report
    - i. Temperature, include a statement on the performance of ERA5 for temperature based on Lopes et al. (2024) Conclusions from this paper are summarized here (attached to deliverable).
    - ii. Precipitation, include a statement on the performance of ERA5 for precipitation based on Lavers et al. (2022) Conclusions from this paper are summarized here (attached to deliverable).
    - iii. Other variable: if drought, base a general statement on temperature and precipitation, and/or perform literature study on the index and region specifically.
- b. For the use of years, using plots produced in Step 3.2.b visually check whether scale (temperature) or dispersion (precipitation) parameter show a trend and only use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid in the sense that the trend is much greater than variability. Otherwise, a non-stationary scale- or
dispersion-parameter would be more appropriate (not part of OAO protocol). Add a sentence on caution on the results should be added to the scientific report Sec. 3.1 - see also Step 3.5b.iv
- c. POTENTIALLY if local knowledge is available: decide on restricting years based on the number of observations over time

## 3.4 decide on which covariate to use
- a. normally this will only be GMST, smoothed with a 4-year running mean to remove ENSO-related variability) and make sure the 4-year smoothed ERA5 GMST is updated up to the current month.
- b. Optional for an in-depth analysis to use a second covariate (e.g., Nino3.4), see also Step 3.7 

C3S defines the 1850–1900 pre‑industrial global mean surface temperature to be 0.88 °C below the 1991–2020 ERA5 global average. 

In [ ]:
client = DataClient(cds_key=your_api_key, beacon_cache_url="https://beacon-era5.maris.nl/")
gmst = client.gmst_xr(time_range=(datetime(year_range[0], 1, 1), datetime.now())) # bbox entire world, end datetime today
gmst_weights = Process.weighted_values(gmst, 't2m') # xarray does not allow as to apply the weight in the same way as pandas/geopandas, therefore we only return the weight and apply it later in the calculation

gmst_monthly = (
    gmst['t2m']
    .groupby('valid_time')
    .map(lambda da: da.weighted(gmst_weights).mean(dim=('latitude', 'longitude')))
    .drop_vars(['number', 'expver'], errors='ignore')
    .to_dataframe(name='gmst')
    .reset_index()
)
gmst_monthly

In [ ]:
if gmst_monthly['valid_time'].iloc[-1].month < 4:
    gmst_monthly = gmst_monthly[gmst_monthly["valid_time"].dt.year != gmst_monthly['valid_time'].iloc[-1].year]
    gmst_yearly = gmst_monthly.groupby(gmst_monthly["valid_time"].dt.year)['gmst'].mean().reset_index()
    gmst_yearly_rolled = Process.calculate_rolling_window(gdf=gmst_yearly, value_col='gmst', datetime_col="valid_time", window=4, min_periods=2, centering=True, method="mean")
    gmst_yearly_rolled = gmst_yearly_rolled.rename(columns={"valid_time": "year"})

else:
    clim31d_xr = xr.open_dataset(os.path.join(your_save_directory, f"climatology_1991-2020_Maximum Temperature.nc"))
    clim31d = gpd.GeoDataFrame(clim31d_xr.to_dataframe().reset_index(), geometry=gpd.points_from_xy(clim31d_xr.to_dataframe().reset_index().longitude, clim31d_xr.to_dataframe().reset_index().latitude), crs="EPSG:4326")
    clim31d = clim31d.rename(columns={"dayofyear": "doy"})
    clim31d['valid_time'] = pd.to_datetime(event_end.year * 1000 + clim31d["doy"], format="%Y%j")
    studyregion = gpd.read_file(os.path.join(your_save_directory, "sf_studyregion.shp"))
    ts_clim31d_studyregion = Process.calculate_seasonal_cycle(clim31d=clim31d, studyregion=studyregion, value_col='t2m', datetime_col="valid_time", event_end=event_end)[0] 
    monthly_clim31d_studyregion = (ts_clim31d_studyregion.resample("ME", on="valid_time")['t2m'].mean().rename_axis("valid_time").reset_index())
    monthly_clim31d_studyregion["valid_time"] = monthly_clim31d_studyregion["valid_time"].dt.to_period("M").dt.to_timestamp()
    gmst_monthly_filled = Process.fill_missing_gmst_with_climatology(gmst_monthly, monthly_clim31d_studyregion, "gmst", "t2m")
    gmst_yearly = gmst_monthly_filled.groupby(gmst_monthly_filled["valid_time"].dt.year)['gmst'].mean().reset_index()
    gmst_yearly_rolled = Process.calculate_rolling_window(gdf=gmst_yearly, value_col='gmst', datetime_col="valid_time", window=4, min_periods=2, centering=True, method="mean")
    gmst_yearly_rolled = gmst_yearly_rolled.rename(columns={"valid_time": "year"})

In [ ]:
merged_gmst = pd.merge(ts_ann_studyregion_subset, gmst_yearly_rolled, left_on="year", right_on="year", how="inner") 
ref_val = merged_gmst.loc[merged_gmst["year"] == event_year, 'gmst'].values[0] # calculate anomaly relative to event year
merged_gmst_anomaly = merged_gmst.copy()
merged_gmst_anomaly["gmst"] = merged_gmst_anomaly['gmst'] - ref_val
merged_gmst_anomaly.to_csv("./data/gmst_t2m.csv", index=False)
merged_gmst_anomaly

## 3.5 Apply statistical method
information for decisions on fit properties:
- a. fit data to statistical model, and check, at least visually, that this fit agrees with the observed data points. Show this in a figure.
    - i. Gauss - appropriate for moderate extremes with low return periods that are not in the tail. Threshold μ (mu), scale parameter σ (sigma). Usually used for seasonal averages. 
    - ii. GEV - appropriate for largest observations from a large sample. Location parameter μ, scale parameter σ, shape parameter ξ. Usually used for 1-day to 14-day averages, annual (or seasonal) maximum
    - iii. If the fit (Gauss or GEV) does not agree well with the observed data points, the other distribution may be tested and used instead.
        - 1. If both fits do not agree well with the observed data points consider changing to an in-depth study to test a Lognormal distribution; possibly Lognormal is better for precipitation (not coded) 

- b. shift/scale with GMST 
    - i. for temperature extremes the distribution shifts due to global warming without changing the shape (μ changes, proportional to the smoothed GMST; σ and ξ are constant).  
    - ii. for precipitation extremes the distribution scales without changing the shape (dispersion parameter (σ/μ) scales, proportional to the smoothed GMST; ξ is constant). 
    - iii. In-depth only (not coded) for precipitation with Lognormal use shift 
    - iv. Use the visual check of scale or dispersion parameter (done in Step 3.3b): use the stationary scale/dispersion-fit if the stationarity assumption is not obviously invalid. Otherwise using a non-stationary scale- or dispersion-parameter would be more appropriate, and a sentence on caution on the results should be added to the scientific report Section. 3.1 
    - v. Include the datapoint of the event under study in the fit 


## <font color='orange'>Please specify the following variables<font>

In [ ]:
%%R -i event_year -i value_col -i fit_type
################# (User selection) ###################
dist = "gev" #choose between "norm" or "gev" 
covnm = "gmst"
lower = FALSE
cooling_offset = 1.3 #fixed to 1.3 for now, can be changed at later stage if required
######################################################

## <font color='green'>Plotting of trends<font>

In [ ]:
%%R -i merged_gmst_anomaly 

df <- merged_gmst_anomaly

mdl <- fit_ns(dist = dist, type = fit_type, data = df, varnm = value_col, covnm = covnm, lower = lower, ev_year = event_year)

# the factual climate should have the GMST of the year in which the event occurred
cov_factual <- data.frame(gmst = df$gmst[df$year == event_year])

# the counterfactual climate can represent any alternative climate (WWA always uses a preindustrial climate, 1.3C cooler than the present)
cov_counterfactual <- data.frame(gmst = df$gmst[df$year == event_year] - cooling_offset)

In [ ]:
bounds = np.array([merged_gmst_anomaly[value_col].min(), merged_gmst_anomaly[value_col].max()])

In [ ]:
%%R -i bounds
# what does the fitted trend look like over time?
# `add_loess = T` will add a nonparametric smoother - use this to check whether the fitted model captures the observed trend
plot_trend(mdl, add_loess = T, ylim = c(bounds[1],bounds[2]))

In [ ]:
%%R -i bounds
# what does the fitted trend look like vs GMST?
plot_covtrend(mdl, xcov = covnm, add_loess = T, ylim = c(bounds[1],bounds[2]))

In [ ]:
%%R -i bounds
# How well does the model fit the data?
# the points should be close to the line - if they're not within the shaded region, the model is a poor fit
plot_returnlevels(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual, xlim = c(1,1000))

In [ ]:
%%R -i bounds
rp_factual <- return_period(mdl, x = bounds[2], fixed_cov = cov_factual)

mdl_est <- cmodel_results (mdl, rp = rp_factual, cov_f = cov_factual, cov_hist = cov_counterfactual)

vals <- mdl_est[1, ]
labs <- colnames(mdl_est)

for(i in seq_along(vals)) {cat(sprintf("%-18s = %g\n", labs[i], vals[i]))}

## <font color='green'>Saving plots</font>

In [ ]:
ylab = (f"{parameter} ({unit})")

In [ ]:
%%R -i ylab
png("./data/fig_obs-trend_era5.png", height = 480, width = 480); {plot_trend(mdl, add_loess = T, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab, lwd = 2)}; invisible(dev.off())
png("./data/fig_obs-gmsttrend_era5.png", height = 480, width = 480); {plot_covtrend(mdl, xcov = "gmst", add_loess = T, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab, lwd = 2)}; invisible(dev.off())
png("./data/fig_obs-returnlevels_era5.png", height = 480, width = 480); {plot_returnlevels(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual, ylim = c(x = bounds[1],x = bounds[2]), ylab = ylab, legend_labels = c("Present climate", "Preindustrial"))}; invisible(dev.off())

## <font color='green'>Saving figure </font>

### Including two plots: 1) the trend over time, 2) GEV fit
#### Includes logos for uptake in report

### <font color="orange"> You can provide titles for both the fitted trend over time and the GEV fit and the figure as a whole </font>

In [ ]:
#################### (User selection) ##################
# Paths to your saved PNGs
file1 = "./data/fig_obs-trend_era5.png"
file2 = "./data/fig_obs-returnlevels_era5.png"
titlefig1 = None
titlefig2 = None
title = None
########################################################

fig, axs, logo_ax = Plot.plot_two_figures(file_left=file1, file_right=file2, add_logos=True, title_fig1=titlefig1, title_fig2=titlefig2, title=title)
fig.savefig(os.path.join(your_save_directory, "trend_analysis_combined.png"), dpi=300, bbox_inches='tight')
print(f"Figure saved to {os.path.join(your_save_directory, 'trend_analysis_combined.png')}") 

## 3.6 Observed probability and trend detection:
- a. Note down decision on the statistical method (see Step 3.5) in the Notes & Tables document
- b. Note dGMST, i.e. the difference between GMSTevent and GMSTpast, based on the preindustrial climate of 1850-1900, in 1 decimal, e.g. 1.3 (this is the value in 2025). C3S defines the 1850–1900 pre‑industrial global mean surface temperature to be 0.88 °C below the 1991–2020 ERA5 global average. C3S will inform us when the dGMST value should be updated - check once a year that this value is still valid (listed in tasklist as an annual task).
- c. Use all observational/reanalysis datasets for the years that passed the quality checks (years see Step 3.2c and 3.3). This is likely ERA5 and potentially also a station data time series.
- d. Note down/save variability or variability over mean (σ or σ/μ) and shape parameter ξ (for GEV) of the fit
- e. Note down return period in the current climate, i.e. in Yevent, e.g., 2025, and decide on the single rounded value of the return period that will be used for communication purposes and the model analyses write this in the output table in Notes & Tables (bottom row). Rounding should correspond to the order of magnitude, e.g., 13.876 (minimum 8.124 to maximum 20.573) could be rounded to 14 (8-21).
    - i. Rounding general guideline:
        - 1. Round to 1 or 2 significant figures
        - 2. Between 10 and 50 round to nearest 5
    - ii. if necessary, make decision on using e.g. lower bound in case of too extreme a return period. Then use this value in the model analysis as well and write a sentence to the scientific report to explain that the best estimate of the return period is too extreme to be useful (>1000 years) and therefore a more meaningful standard return period of e.g. 1000 years has been used for the analysis.
- f. Note down the threshold value corresponding to the event in the first column of the table in brackets (value).
- g. Note down probability ratio, PR, for dGMST, and change in intensity, ΔI, (absolute value for temperature, relative % change for precipitation, i.e., (present-past)/past x 100%.
- h. In the case of a PR <1, which will most often occur for cold extremes which are becoming less likely with climate change, the inverse PR (as well as inverse uncertainty range) should also be noted – the inverse PR will be calculated in Step 6.6a.i. Statements based on the inverse PR (values larger than 1) are much easier to communicate, see Step 6.9a.i. 

In [ ]:
%%R
# use the built-in function to bootstrap the model results

boot_res <- boot_ci(mdl, cov_f = cov_factual, cov_cf = cov_counterfactual)
# these two lines might not even be needed
colnames(boot_res)[colnames(boot_res) == "2.5%"]  <- "lower"
colnames(boot_res)[colnames(boot_res) == "97.5%"] <- "upper"

# transpose to get the correct output format
boot_res_t <- data.frame(
    t(
        unlist(
            lapply(rownames(boot_res), function(rn) {
                setNames(
                    as.numeric(boot_res[rn, ]),
                    paste0(rn, "_", colnames(boot_res))
                )
            })
        )
    ),
    row.names = "era5"
)
boot_res_t

# save as a .csv to look at them later
# this .csv is used for the Synthesis notebook
write.csv(boot_res_t, "./data/res-obs_era5.csv")

In [ ]:
%%R -i bounds
rp_factual <- return_period(mdl, x = bounds[2], fixed_cov = cov_factual)
rp_counterfactual <- return_period(mdl, x = bounds[2], fixed_cov = cov_counterfactual)

cat(sprintf("Return period (factual): %f (%f – %f)\n", rp_factual, boot_res[19], boot_res[31]))
cat(sprintf("Return period (counterfactual): %f\n", rp_counterfactual))

In [ ]:
%%R

# try fitting different distributions to compare which actually fits better
mdl_gev <- fit_ns(dist = "gev", type = "shift", data = df, varnm = value_col, covnm = "gmst", lower = F)
mdl_norm <- fit_ns(dist = "norm", type = "shift", data = df, varnm = value_col, covnm = "gmst", lower = F)

# use AIC to identify the model that fits better (lower score is better)
cat(sprintf("AIC (GEV): %f\n", aic(mdl_gev)))
cat(sprintf("AIC (Normal): %f\n", aic(mdl_norm)))